## Ingestão e Tratamento dos Dados

In [2]:
#Importação dos dados e da biblioteca básica

import pandas as pd
import unicodedata
import re

df_infraestruturas = pd.read_csv(
    'estimativaAnoQtdeHospitais(in).csv',
    encoding='latin1',
    sep=';'
)

df_populacao = pd.read_csv(
    'estimativaMunicipiosMG(in).csv',
    encoding='latin1',
    sep=';'
)

In [3]:
#Verificando as colunas
print(df_infraestruturas.columns)
print(df_populacao.columns)

Index(['Município/Ano', '2015', '2016', '2017', '2018', '2019', '2020', '2021',
       '2022', '2023', '2024', '2025'],
      dtype='object')
Index(['Município/Ano', '2015', '2016', '2017', '2018', '2019', '2020', '2021',
       '2024', '2025'],
      dtype='object')


In [4]:
#Renoameando coluna principal

df_populacao = df_populacao.rename(columns={'Município/Ano': 'municipio'})
df_infraestruturas = df_infraestruturas.rename(columns={'Município/Ano': 'municipio'})

In [5]:
#Transformação das colunas dos anos em variáveis de uma única coluna
df_populacao_long = df_populacao.melt(
    id_vars='municipio',
    var_name='ano',
    value_name='populacao'
)
df_infraestruturas_long = df_infraestruturas.melt(
    id_vars='municipio',
    var_name='ano',
    value_name='infraestruturas'
)

In [6]:
#Vizualização das tabelas
df_populacao_long.head()

,municipio,ano,populacao
0,Abadia dos Dourados (MG),2015,7015.0
1,Abaeté (MG),2015,23535.0
2,Abre Campo (MG),2015,13719.0
3,Acaiaca (MG),2015,4056.0
4,Açucena (MG),2015,10140.0


In [7]:
df_infraestruturas_long.head()

,municipio,ano,infraestruturas
0,310010 ABADIA DOS DOURADOS,2015,6.0
1,310020 ABAETE,2015,37.0
2,310030 ABRE CAMPO,2015,12.0
3,310040 ACAIACA,2015,6.0
4,310050 ACUCENA,2015,9.0


In [8]:
#Converter a coluna "ano" para valor inteiro 

df_populacao_long['ano'] = df_populacao_long['ano'].astype(int)
df_infraestruturas_long['ano'] = df_infraestruturas_long['ano'].astype(int)

In [9]:
#Converter as colunas municipio para string
df_populacao_long['municipio'] = df_populacao_long['municipio'].astype(str)
df_infraestruturas_long['municipio'] = df_infraestruturas_long['municipio'].astype(str)

In [10]:
#Função que remove e padronizar as variáveis
def limpar_municipio_ibge(nome):
    # remove "(MG)"
    nome = nome.replace("(MG)", "")
    
    # remove espaços extras
    nome = nome.strip()
    
    # coloca em maiúsculo
    nome = nome.upper()
    
    # remove acentos
    nome = ''.join(
        c for c in unicodedata.normalize('NFD', nome)
        if unicodedata.category(c) != 'Mn'
    )
    
    return nome
#Aplica a função no dataframe
df_populacao_long['municipio'] = df_populacao_long['municipio'].apply(limpar_municipio_ibge)

In [11]:
df_populacao_long.head()

,municipio,ano,populacao
0,ABADIA DOS DOURADOS,2015,7015.0
1,ABAETE,2015,23535.0
2,ABRE CAMPO,2015,13719.0
3,ACAIACA,2015,4056.0
4,ACUCENA,2015,10140.0


In [12]:
#Remove o código do IBGE que está como prefixo
def remover_codigo_municipios(valor):
    #Verifica se é uma tupla
    if isinstance(valor, tuple):
        valor = valor[0]
    
    partes = valor.split(' ', 1)
    return partes[1] if len(partes) > 1 else valor

#Aplica a função no dataframe
df_infraestruturas_long['municipio'] =df_infraestruturas_long['municipio'].apply(remover_codigo_municipios)

In [13]:
df_infraestruturas_long.head()

,municipio,ano,infraestruturas
0,ABADIA DOS DOURADOS,2015,6.0
1,ABAETE,2015,37.0
2,ABRE CAMPO,2015,12.0
3,ACAIACA,2015,6.0
4,ACUCENA,2015,9.0


In [14]:
#Faltam alguns anos na tabela do DATASUS, por isso, foi feita uma interseção para usar apenas os anos que tivessem dados
anos_comuns = set(df_populacao_long['ano']).intersection(set(df_infraestruturas_long['ano']))

df_populacao_long = df_populacao_long[df_populacao_long['ano'].isin(anos_comuns)]
df_infraestruturas_long =df_infraestruturas_long[df_infraestruturas_long['ano'].isin(anos_comuns)]

In [15]:
#Verificação das variáveis disponíveis na coluna ano
print("Anos população:", sorted(df_populacao_long['ano'].unique()))
print("Anos hospitais:", sorted(df_infraestruturas_long['ano'].unique()))

Anos população: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2024, 2025]
Anos hospitais: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2024, 2025]


In [16]:
#integração dos dados
df_principal = pd.merge(
    df_populacao_long,
    df_infraestruturas_long,
    on=['municipio', 'ano'],
    how='inner'
)

In [17]:
df_principal.head()

,municipio,ano,populacao,infraestruturas
0,ABADIA DOS DOURADOS,2015,7015.0,6.0
1,ABAETE,2015,23535.0,37.0
2,ABRE CAMPO,2015,13719.0,12.0
3,ACAIACA,2015,4056.0,6.0
4,ACUCENA,2015,10140.0,9.0


## Criação dos indicadores

In [19]:
#Ordenação dos valores por município e ano para fazer os 
# indicadores baseados na tupla anterior caso possuam o mesmo município
df_principal = df_principal.sort_values(['municipio', 'ano'])

# Indicador de habitantes por infraestrutura de saúde
df_principal['habitantes_por_infraestrutura'] = (df_principal['populacao'] / df_principal['infraestruturas'])

#Crescimento Populacional
df_principal['crescimento_populacional'] = df_principal.groupby('municipio')['populacao'].diff()

#Crescimento das infraestruturas
df_principal['crescimento_infraestruturas'] = df_principal.groupby('municipio')['infraestruturas'].diff()

#Mostra as primeira 8 linhas para avaliar se o cálculo do primeiro município da lista foi coerente
df_principal.head(8)

,municipio,ano,populacao,infraestruturas,habitantes_por_infraestrutura,crescimento_populacional,crescimento_infraestruturas
0,ABADIA DOS DOURADOS,2015,7015.0,6.0,1169.166667,NaN,NaN
850,ABADIA DOS DOURADOS,2016,7037.0,6.0,1172.833333,22.0,0.0
1700,ABADIA DOS DOURADOS,2017,7059.0,5.0,1411.800000,22.0,-1.0
2550,ABADIA DOS DOURADOS,2018,6972.0,5.0,1394.400000,-87.0,0.0
3400,ABADIA DOS DOURADOS,2019,6989.0,5.0,1397.800000,17.0,0.0
4250,ABADIA DOS DOURADOS,2020,7006.0,6.0,1167.666667,17.0,1.0
5100,ABADIA DOS DOURADOS,2021,7022.0,6.0,1170.333333,16.0,0.0
5950,ABADIA DOS DOURADOS,2024,6365.0,8.0,795.625000,-657.0,2.0


In [20]:
#Dowload da pasta para fazer dashboard
df_principal.to_excel('base_final.xlsx', index=False)